# Essay I — Panel Pipeline

Builds a firm-year panel from the Thesis 3 (or Thesis 2) variance decomposition and quarterly firm controls, then estimates firm + year fixed-effects regressions and saves a standard regression output table.

In [1]:
import sys
from pathlib import Path

# Add Scripts folder so local modules resolve correctly.
SCRIPTS_DIR = Path(".").resolve()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS
pd.set_option("display.float_format", "{:.6f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

PROJECT_DIR = SCRIPTS_DIR.parent
DATA_DIR    = PROJECT_DIR / "Data"
OUTPUTS_DIR = PROJECT_DIR / "Outputs" / "essay1_pipeline"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Import daily-sheet helpers directly (parse_data_workbook also loads quarterly_,
# which has a non-standard format — we parse each sheet separately below).
from data_interface import (read_bloomberg_two_row_sheet,
                             split_price_volume,
                             build_daily_canonical_panel)
from Thesis_3_core import run_thesis3_from_daily_panel
from Thesis_2    import run_thesis2_from_daily_panel

print("Imports OK")
print(f"PROJECT_DIR : {PROJECT_DIR}")


Imports OK
PROJECT_DIR : /Users/docx/VCS/Thesis


## Configuration

Edit `OUTCOME_VARS` and `CONTROL_VARS` to change what the regression estimates. All other settings are also here.

In [2]:
# ================================================================
# REGRESSION CONFIGURATION — edit these to set up your analysis
# ================================================================

# Dependent variable(s): variance-share columns from the decomposition.
# thesis3 options : "PrivateInfoShare", "PublicInfoShare", "MktInfoShare",
#                   "NoiseShare", "FirmInfoShareAgg"
# thesis2 options : "FirmInfoShare", "MktInfoShare", "NoiseShare"
OUTCOME_VARS: list = [
    "PrivateInfoShare",
    "PublicInfoShare",
    "MktInfoShare",
    "NoiseShare",
]

# Independent variables: raw column names from the quarterly sheet of data.xlsx.
# Each is aggregated to a yearly mean before entering the regression.
#
# Available columns and their coverage in the current data.xlsx:
#   ESG_SCORE            84%   ENVIRONMENTAL_SCORE  84%
#   SOCIAL_SCORE         84%   GOVERNANCE_SCORE     85%
#   CUR_MKT_CAP          95%   VOLATILITY_90D       94%
#   PCT_INSIDER_SHARES   93%   FNCL_LVRG            55%
#   HEADLINE_BVPS        61%   RETURN_COM_EQY       54%
#   BS_TOT_ASSET         62%   HEADLINE_CAPEX       49%
#   PROF_MARGIN          60%   ASSET_TURNOVER       57%
#   CASH_FLOW_PER_SH     51%
CONTROL_VARS: list = [
    "ESG_SCORE",
    "CUR_MKT_CAP",
    "VOLATILITY_90D",
    "FNCL_LVRG",
]

# ---- pipeline settings ----
DECOMP_MODEL    = "thesis3"          # "thesis3" or "thesis2"
START_YEAR      = 2015
END_YEAR        = 2025
DATA_FILE       = DATA_DIR / "data.xlsx"
DAILY_SHEET     = "daily_"
MARKET_TICKER   = "SXXP Index"
MAX_STOCKS      = None               # set e.g. to 20 for a quick smoke test

# ================================================================
print(f"Y      : {OUTCOME_VARS}")
print(f"X      : {CONTROL_VARS}")
print(f"Model  : {DECOMP_MODEL}  |  {START_YEAR}–{END_YEAR}")
print(f"Stocks : {'all' if MAX_STOCKS is None else MAX_STOCKS}")


Y      : ['PrivateInfoShare', 'PublicInfoShare', 'MktInfoShare', 'NoiseShare']
X      : ['ESG_SCORE', 'CUR_MKT_CAP', 'VOLATILITY_90D', 'FNCL_LVRG']
Model  : thesis3  |  2015–2025
Stocks : all


## Helper functions

In [3]:
def _safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")


def standardize_decomposition_output(raw_results, decomp_model):
    df = raw_results.copy()
    if "period" in df.columns and "year" not in df.columns:
        df["year"] = pd.to_numeric(df["period"], errors="coerce")
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["decomp_model"] = decomp_model
    df["noise_share_proxy"] = pd.to_numeric(df["NoiseShare"], errors="coerce") if "NoiseShare" in df.columns else np.nan
    if "VarTotal" in df.columns:
        df["total_var"] = pd.to_numeric(df["VarTotal"], errors="coerce")
    elif "TotalVar" in df.columns:
        df["total_var"] = pd.to_numeric(df["TotalVar"], errors="coerce")
    else:
        df["total_var"] = np.nan
    for col in ["MktInfoShare", "FirmInfoShare", "PrivateInfoShare", "PublicInfoShare", "NoiseShare"]:
        if col not in df.columns:
            df[col] = np.nan
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["FirmInfoShareAgg"] = (df["PrivateInfoShare"] + df["PublicInfoShare"]) if decomp_model == "thesis3" else np.nan
    base = ["stock","year","decomp_model","noise_share_proxy","total_var",
            "MktInfoShare","FirmInfoShare","PrivateInfoShare","PublicInfoShare","FirmInfoShareAgg","NoiseShare"]
    other = [c for c in df.columns if c not in base]
    return df[base + other].copy()


def aggregate_quarterly_controls_to_year(quarterly_df, control_vars):
    df = quarterly_df.copy()
    for col in control_vars:
        if col in df.columns:
            df[col] = _safe_numeric(df[col])
    grouped = df.groupby(["stock", "year"])
    yearly  = grouped.size().reset_index(name="n_quarters")
    for col in control_vars:
        target = f"{col}_y"
        if col in df.columns:
            stats  = grouped[col].mean().reset_index(name=target)
            yearly = yearly.merge(stats, on=["stock","year"], how="left")
        else:
            yearly[target] = np.nan
    if "GICS_INDUSTRY" in df.columns:
        gics = (df.dropna(subset=["GICS_INDUSTRY"])
                  .sort_values(["stock","year","date"])
                  .groupby(["stock","year"], as_index=False).first()
                  [["stock","year","GICS_INDUSTRY"]]
                  .rename(columns={"GICS_INDUSTRY": "gics_industry_y"}))
        yearly = yearly.merge(gics, on=["stock","year"], how="left")
    else:
        yearly["gics_industry_y"] = np.nan
    return yearly


def apply_sample_filters(panel_df, start_year, end_year):
    df = panel_df.copy()
    df = df[df["year"].between(start_year, end_year, inclusive="both")]
    for col in ["MktInfoShare","FirmInfoShare","PrivateInfoShare","PublicInfoShare","FirmInfoShareAgg","NoiseShare"]:
        valid = df[col].isna() | ((df[col] >= 0) & (df[col] <= 100))
        df = df[valid]
    df = df[df["noise_share_proxy"].notna()]
    if "total_var" in df.columns:
        df = df[(df["total_var"].isna()) | (df["total_var"] > 0)]
    return df.reset_index(drop=True)


print("Helper functions defined")


Helper functions defined


## Regression functions

In [4]:
def _fit_clustered_fe_panel(df, outcome_col, control_cols):
    yearly_cols = [f"{c}_y" for c in control_cols if f"{c}_y" in df.columns]
    reg_df = df[["stock","year",outcome_col] + yearly_cols].copy()
    for col in [outcome_col] + yearly_cols:
        reg_df[col] = pd.to_numeric(reg_df[col], errors="coerce")
    reg_df = reg_df.dropna()

    _empty = dict(outcome=outcome_col, controls=control_cols,
                  n_obs=0, n_stocks=0, n_years=0,
                  multiple_r=np.nan, r_square=np.nan, adj_r_square=np.nan,
                  std_error_regression=np.nan,
                  df_reg=np.nan, df_resid=np.nan, df_total=np.nan,
                  ss_reg=np.nan, ss_resid=np.nan, ss_total=np.nan,
                  ms_reg=np.nan, ms_resid=np.nan,
                  f_stat=np.nan, f_pval=np.nan, coef_table=[], error=None)

    if len(reg_df) == 0 or reg_df["stock"].nunique() < 2 or reg_df["year"].nunique() < 2:
        _empty.update(n_obs=int(len(reg_df)),
                      n_stocks=int(reg_df["stock"].nunique()) if len(reg_df) > 0 else 0,
                      n_years=int(reg_df["year"].nunique()) if len(reg_df) > 0 else 0,
                      error="Insufficient variation for FE estimation.")
        return _empty

    reg_df = reg_df.set_index(["stock","year"]).sort_index()
    y, x = reg_df[outcome_col], reg_df[yearly_cols]

    try:
        fitted = PanelOLS(dependent=y, exog=x,
                          entity_effects=True, time_effects=True,
                          drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)
        n, k = int(fitted.nobs), len(fitted.params)
        ss_resid  = float((fitted.resids ** 2).sum())
        r2_within = float(fitted.rsquared_within)
        ss_total  = ss_resid / (1.0 - r2_within) if r2_within < 1.0 else np.nan
        ss_reg    = (ss_total - ss_resid) if not np.isnan(ss_total) else np.nan
        df_reg    = k
        df_resid  = int(fitted.df_resid)
        ms_reg    = ss_reg  / df_reg   if df_reg  > 0 and not np.isnan(ss_reg)  else np.nan
        ms_resid  = ss_resid / df_resid if df_resid > 0 else np.nan
        rmse      = float(np.sqrt(ms_resid)) if not np.isnan(ms_resid) else np.nan
        adj_r2    = float(1.0 - (1.0 - r2_within) * (n-1) / (n-k-1)) if (n-k-1) > 0 else np.nan
        try:    f_stat, f_pval = float(fitted.f_statistic.stat), float(fitted.f_statistic.pval)
        except: f_stat, f_pval = np.nan, np.nan
        ci = fitted.conf_int()
        coef_table = [{"variable": v,
                       "coef": float(fitted.params[v]),
                       "se":   float(fitted.std_errors[v]),
                       "t_stat":  float(fitted.tstats[v]),
                       "p_value": float(fitted.pvalues[v]),
                       "lower_95": float(ci.loc[v,"lower"]),
                       "upper_95": float(ci.loc[v,"upper"])} for v in fitted.params.index]
        return dict(outcome=outcome_col, controls=control_cols,
                    n_obs=n, n_stocks=int(reg_df.index.get_level_values(0).nunique()),
                    n_years=int(reg_df.index.get_level_values(1).nunique()),
                    multiple_r=float(np.sqrt(r2_within)), r_square=r2_within,
                    adj_r_square=adj_r2, std_error_regression=rmse,
                    df_reg=df_reg, df_resid=df_resid, df_total=n-1,
                    ss_reg=ss_reg, ss_resid=ss_resid, ss_total=ss_total,
                    ms_reg=ms_reg, ms_resid=ms_resid,
                    f_stat=f_stat, f_pval=f_pval, coef_table=coef_table, error=None)
    except Exception as exc:
        r = dict(_empty)
        r.update(n_obs=int(len(reg_df)),
                 n_stocks=int(reg_df.index.get_level_values(0).nunique()),
                 n_years=int(reg_df.index.get_level_values(1).nunique()),
                 error=str(exc))
        return r


def run_fe_regressions(panel_df, outcome_vars, control_vars):
    available = [c for c in outcome_vars if c in panel_df.columns]
    return [_fit_clustered_fe_panel(panel_df, o, control_vars) for o in available]


def _rows_for_block(result):
    out, controls_str = result["outcome"], " + ".join(f"{c}_y" for c in result["controls"])
    def _f(v, d=6):
        if v is None or (isinstance(v, float) and np.isnan(v)): return ""
        if isinstance(v, (int, np.integer)): return int(v)
        return round(float(v), d)
    rows = []
    rows += [[f"Regression: {out} ~ {controls_str}","","","","","",""],
             ["Model: PanelOLS  |  Entity + time FE  |  Stock-clustered SE","","","","","",""],
             [""]]
    rows += [["Regression Statistics"],
             ["Multiple R",        _f(result["multiple_r"])],
             ["R Square (within)", _f(result["r_square"])],
             ["Adjusted R Square", _f(result["adj_r_square"])],
             ["Standard Error",    _f(result["std_error_regression"])],
             ["Observations",      result["n_obs"]],
             ["Unique stocks",     result["n_stocks"]],
             ["Years",             result["n_years"]]]
    if result.get("error"):
        rows.append(["Note", result["error"]])
    rows.append([""])
    rows += [["ANOVA","","","","",""],
             ["","df","SS","MS","F","Significance F"],
             ["Regression", result["df_reg"],   _f(result["ss_reg"]),   _f(result["ms_reg"]),   _f(result["f_stat"]), _f(result["f_pval"])],
             ["Residual",   result["df_resid"], _f(result["ss_resid"]), _f(result["ms_resid"]), "",""],
             ["Total",      result["df_total"], _f(result["ss_total"]), "","",""],
             [""]]
    rows.append(["","Coefficients","Standard Error","t Stat","P-value","Lower 95%","Upper 95%"])
    for r in result["coef_table"]:
        rows.append([r["variable"],_f(r["coef"]),_f(r["se"]),_f(r["t_stat"]),_f(r["p_value"]),_f(r["lower_95"]),_f(r["upper_95"])])
    rows += [[""],  [""]]
    return rows


def save_regression_table(results, path):
    all_rows = []
    for r in results:
        all_rows.extend(_rows_for_block(r))
    w = max((len(r) for r in all_rows), default=7)
    pd.DataFrame([r + [""]*(w-len(r)) for r in all_rows]).to_csv(path, index=False, header=False)


print("Regression functions defined")


Regression functions defined


## Step 1 — Run variance decomposition

Loads the daily returns panel from `daily_` and runs the selected decomposition model. Takes ~1–2 minutes for the full stock universe.

In [5]:
import time

print(f"Loading daily_ sheet…")
t0 = time.time()
_daily_raw = read_bloomberg_two_row_sheet(DATA_FILE, sheet_name=DAILY_SHEET)
_parts     = split_price_volume(_daily_raw)
daily      = build_daily_canonical_panel(_parts["prices"], _parts["volume"],
                                         market_ticker=MARKET_TICKER)
print(f"  {daily['stock'].nunique()} stocks, {len(daily):,} rows  ({time.time()-t0:.1f}s)")

if MAX_STOCKS is not None:
    keep = daily["stock"].drop_duplicates().head(MAX_STOCKS)
    daily = daily[daily["stock"].isin(keep)].copy()
    print(f"  Capped to {daily['stock'].nunique()} stocks (MAX_STOCKS={MAX_STOCKS})")

print(f"Running {DECOMP_MODEL} decomposition…")
t1 = time.time()
if DECOMP_MODEL == "thesis3":
    _decomp_raw = run_thesis3_from_daily_panel(daily)
else:
    _decomp_raw = run_thesis2_from_daily_panel(daily)

decomp_results = standardize_decomposition_output(_decomp_raw["results"], DECOMP_MODEL)
ew_df = _decomp_raw.get("ew", pd.DataFrame())
vw_df = _decomp_raw.get("vw", pd.DataFrame())

print(f"  Done in {time.time()-t1:.1f}s")
print(f"  Decomp results: {decomp_results.shape[0]:,} rows, "
      f"{decomp_results['stock'].nunique()} stocks, "
      f"years {int(decomp_results['year'].min())}–{int(decomp_results['year'].max())}")
decomp_results.head(3)


Loading daily_ sheet…


  596 stocks, 1,617,457 rows  (24.8s)
Running thesis3 decomposition…


  Done in 24.0s
  Decomp results: 6,232 rows, 595 stocks, years 2015–2025


,stock,year,decomp_model,noise_share_proxy,total_var,MktInfoShare,FirmInfoShare,PrivateInfoShare,PublicInfoShare,FirmInfoShareAgg,NoiseShare,MktInfo,PrivateInfo,PublicInfo,Noise,k_ar,max_eigenvalue,n_obs,VarTotal
0,A2A IM Equity,2015,thesis3,11.902496,15633.653889,29.498304,NaN,38.955959,19.643242,58.599201,11.902496,4611.662728,6090.239797,3070.956410,1860.794953,5,0.295110,260,15633.653889
1,A2A IM Equity,2016,thesis3,11.792327,15951.595426,39.036022,NaN,32.560400,16.611252,49.171652,11.792327,6226.868225,5193.903260,2649.759664,1881.064277,5,0.230659,261,15951.595426
2,A2A IM Equity,2017,thesis3,23.102584,7886.409192,1.530873,NaN,66.348309,9.018235,75.366543,23.102584,120.730887,5232.499102,711.214898,1821.964305,5,0.339091,260,7886.409192


## Step 2 — Load and aggregate quarterly controls

The quarterly data is split across two sheets: tickers in `quarterly` (row 3) and values in `quarterly_` (rows 1–44). We parse both directly and join on column position.

In [6]:
def _parse_excel_dates(series):
    """Convert Excel serial-day numbers or strings to datetime."""
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().mean() > 0.8:
        return pd.to_datetime(numeric, unit="D", origin="1899-12-30", errors="coerce")
    return pd.to_datetime(series, errors="coerce")


def read_quarterly_panel(data_file, ticker_sheet="quarterly", data_sheet="quarterly_"):
    """
    Parse quarterly firm data from two complementary sheets.

    Ticker names: `ticker_sheet` row 3 (forward-filled).
    Field codes + values: `data_sheet` row 0 (header) and rows 1+ (data).
    Returns canonical long-format: stock, date, year, <fields>.
    """
    q_hdr  = pd.read_excel(data_file, sheet_name=ticker_sheet,  header=None)
    q_data = pd.read_excel(data_file, sheet_name=data_sheet,    header=None)

    tickers    = q_hdr.iloc[3].ffill().values     # length = n_cols
    fields     = q_data.iloc[0].values            # length = n_cols
    dates      = _parse_excel_dates(q_data.iloc[1:, 0])
    data_block = q_data.iloc[1:, 1:].reset_index(drop=True)
    ticker_arr = np.array(tickers[1:])
    field_arr  = np.array(fields[1:])

    rows = []
    for ticker in pd.Series(ticker_arr).dropna().unique():
        mask  = ticker_arr == ticker
        chunk = data_block.iloc[:, mask].copy()
        chunk.columns = field_arr[mask]
        chunk["stock"] = ticker
        chunk["date"]  = dates.values
        rows.append(chunk)

    q = pd.concat(rows, ignore_index=True)
    q["year"] = pd.to_datetime(q["date"]).dt.year
    id_cols = ["stock","date","year"]
    return q[id_cols + [c for c in q.columns if c not in id_cols]].sort_values(["stock","date"]).reset_index(drop=True)


print("Reading quarterly data…")
quarterly = read_quarterly_panel(DATA_FILE)

if MAX_STOCKS is not None:
    keep_stocks = decomp_results["stock"].drop_duplicates().head(MAX_STOCKS)
    quarterly = quarterly[quarterly["stock"].isin(keep_stocks)].copy()

controls_yearly = aggregate_quarterly_controls_to_year(quarterly, CONTROL_VARS)

print(f"Quarterly rows : {len(quarterly):,}  |  stocks: {quarterly['stock'].nunique()}")
print(f"Date range     : {quarterly['date'].min().date()} – {quarterly['date'].max().date()}")
print(f"Yearly controls: {len(controls_yearly):,} firm-years, "
      f"{controls_yearly['stock'].nunique()} stocks")

# Verify overlap with decomp stocks
overlap = set(decomp_results['stock'].unique()) & set(controls_yearly['stock'].unique())
print(f"Overlap (decomp ∩ controls): {len(overlap)} stocks")
controls_yearly.head(3)


Reading quarterly data…


Quarterly rows : 26,268  |  stocks: 597
Date range     : 2015-03-31 – 2025-12-31
Yearly controls: 6,567 firm-years, 597 stocks
Overlap (decomp ∩ controls): 595 stocks


,stock,year,n_quarters,ESG_SCORE_y,CUR_MKT_CAP_y,VOLATILITY_90D_y,FNCL_LVRG_y,gics_industry_y
0,A2A IM Equity,2015,4,3.970000,3447.370650,28.674750,3.754250,NaN
1,A2A IM Equity,2016,4,4.022500,3765.752125,29.872250,3.680875,NaN
2,A2A IM Equity,2017,4,4.230000,4596.755175,19.428750,3.570775,NaN


## Step 3 — Merge, filter, and regress

In [7]:
merged         = decomp_results.merge(controls_yearly, on=["stock","year"], how="left")
filtered_panel = apply_sample_filters(merged, start_year=START_YEAR, end_year=END_YEAR)

print(f"Merged panel   : {len(merged):,} rows")
print(f"Filtered panel : {len(filtered_panel):,} rows, "
      f"{filtered_panel['stock'].nunique()} stocks, "
      f"{filtered_panel['year'].nunique()} years  "
      f"({int(filtered_panel['year'].min())}–{int(filtered_panel['year'].max())})")
print(f"ESG coverage   : {filtered_panel['ESG_SCORE_y'].notna().sum()} obs with ESG_SCORE")

# Run FE regressions.
results = run_fe_regressions(filtered_panel, OUTCOME_VARS, CONTROL_VARS)

# Save outputs.
filtered_panel.to_csv(OUTPUTS_DIR / f"essay1_panel_{DECOMP_MODEL}.csv", index=False)
table_path = OUTPUTS_DIR / f"essay1_regression_table_{DECOMP_MODEL}.csv"
save_regression_table(results, table_path)

print(f"\nSaved panel          : {OUTPUTS_DIR / f'essay1_panel_{DECOMP_MODEL}.csv'}")
print(f"Saved regression table: {table_path}")


Merged panel   : 6,232 rows
Filtered panel : 6,232 rows, 595 stocks, 11 years  (2015–2025)
ESG coverage   : 5849 obs with ESG_SCORE



Saved panel          : /Users/docx/VCS/Thesis/Outputs/essay1_pipeline/essay1_panel_thesis3.csv
Saved regression table: /Users/docx/VCS/Thesis/Outputs/essay1_pipeline/essay1_regression_table_thesis3.csv


## Results

In [8]:
def _display_block(r):
    out = r["outcome"]
    controls_str = " + ".join(f"{c}_y" for c in r["controls"])
    title = f"Regression: {out} ~ {controls_str}"
    print(f"\n{'='*len(title)}")
    print(title)
    print(f"Model: PanelOLS  |  Entity + time FE  |  Stock-clustered SE")
    print(f"{'='*len(title)}")

    print("\nRegression Statistics")
    for label, val in [
        ("Multiple R",        r["multiple_r"]),
        ("R Square (within)", r["r_square"]),
        ("Adjusted R Square", r["adj_r_square"]),
        ("Standard Error",    r["std_error_regression"]),
        ("Observations",      r["n_obs"]),
        ("Unique stocks",     r["n_stocks"]),
        ("Years",             r["n_years"]),
    ]:
        fmt = f"{val:.6f}" if isinstance(val, float) and not np.isnan(val) else str(val)
        print(f"  {label:<22} {fmt}")
    if r.get("error"):
        print(f"  NOTE: {r['error']}")

    print("\nANOVA")
    anova = pd.DataFrame([
        {"Source":"Regression","df":r["df_reg"],  "SS":r["ss_reg"],  "MS":r["ms_reg"],  "F":r["f_stat"],"Sig.F":r["f_pval"]},
        {"Source":"Residual",  "df":r["df_resid"],"SS":r["ss_resid"],"MS":r["ms_resid"],"F":"",         "Sig.F":""},
        {"Source":"Total",     "df":r["df_total"],"SS":r["ss_total"],"MS":"",            "F":"",         "Sig.F":""},
    ])
    print(anova.to_string(index=False))

    if r["coef_table"]:
        print("\nCoefficients")
        coef_df = pd.DataFrame(r["coef_table"])[
            ["variable","coef","se","t_stat","p_value","lower_95","upper_95"]
        ].rename(columns={"variable":"","coef":"Coeff.",
                          "se":"Std.Err.","t_stat":"t Stat",
                          "p_value":"P-value","lower_95":"Lower 95%","upper_95":"Upper 95%"})
        print(coef_df.to_string(index=False))
    print()


for r in results:
    _display_block(r)



Regression: PrivateInfoShare ~ ESG_SCORE_y + CUR_MKT_CAP_y + VOLATILITY_90D_y + FNCL_LVRG_y
Model: PanelOLS  |  Entity + time FE  |  Stock-clustered SE

Regression Statistics
  Multiple R             nan
  R Square (within)      -0.003018
  Adjusted R Square      -0.004202
  Standard Error         11.762043
  Observations           3395
  Unique stocks          345
  Years                  11

ANOVA
    Source   df            SS          MS        F    Sig.F
Regression    4  -1263.941435 -315.985359 0.462245 0.763502
  Residual 3036 420017.425151  138.345660                  
     Total 3394 418753.483715                              

Coefficients
                    Coeff.  Std.Err.    t Stat  P-value  Lower 95%  Upper 95%
     ESG_SCORE_y -0.183082  0.474817 -0.385584 0.699831  -1.114077   0.747913
   CUR_MKT_CAP_y -0.000004  0.000004 -1.084019 0.278442  -0.000011   0.000003
VOLATILITY_90D_y -0.012342  0.032121 -0.384241 0.700827  -0.075324   0.050639
     FNCL_LVRG_y -0.000756  0.